# Cleaned Data Review

## Project

RetailOps Insight — Retail Operations Analytics, Data Quality, and Support Dashboard

## Purpose

This notebook reviews the cleaned retail transaction data produced by the data cleaning pipeline.

The goal is to confirm that the cleaned outputs are usable for analysis, reporting, KPI calculation, and dashboard development.

## Questions Reviewed

- Do the cleaned files load correctly?
- Do row counts match earlier cleaning and SQL outputs?
- What is the transaction status distribution?
- How much completed sales, cancellation value, and data quality risk exists?
- How much data is affected by missing customer IDs?
- How many rows are flagged as duplicates?

In [1]:
from pathlib import Path

import pandas as pd


In [2]:
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

cleaned_dir = project_root / "data" / "cleaned"
report_dir = project_root / "reports" / "data_quality"
sql_output_dir = project_root / "reports" / "sql_outputs"

print("Current directory:", current_dir)
print("Project root:", project_root)
print("Cleaned data directory:", cleaned_dir)
print("Data quality report directory:", report_dir)
print("SQL output directory:", sql_output_dir)

Current directory: /home/tirtha/Projects/retailops-insight/notebooks
Project root: /home/tirtha/Projects/retailops-insight
Cleaned data directory: /home/tirtha/Projects/retailops-insight/data/cleaned
Data quality report directory: /home/tirtha/Projects/retailops-insight/reports/data_quality
SQL output directory: /home/tirtha/Projects/retailops-insight/reports/sql_outputs


In [3]:
transactions_file = cleaned_dir / "online_retail_transactions_cleaned_sample.csv"
products_file = cleaned_dir / "products_cleaned.csv"
customers_file = cleaned_dir / "customers_cleaned.csv"
cleaning_summary_file = report_dir / "cleaning_summary.csv"
data_quality_issues_file = report_dir / "data_quality_issues_sample.csv"

transactions = pd.read_csv(transactions_file)
products = pd.read_csv(products_file)
customers = pd.read_csv(customers_file)
cleaning_summary = pd.read_csv(cleaning_summary_file)
data_quality_issues = pd.read_csv(data_quality_issues_file)

print("Files loaded successfully.")
print("Transactions sample shape:", transactions.shape)
print("Products shape:", products.shape)
print("Customers shape:", customers.shape)
print("Cleaning summary shape:", cleaning_summary.shape)
print("Data quality issues sample shape:", data_quality_issues.shape)

Files loaded successfully.
Transactions sample shape: (10000, 27)
Products shape: (4070, 8)
Customers shape: (4372, 7)
Cleaning summary shape: (17, 2)
Data quality issues sample shape: (10000, 9)


## Preview Cleaned Transactions

The cleaned transaction table contains standardized column names, transaction status labels, reporting fields, and data quality flags.

In [6]:
transactions.head()

,raw_row_id,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,has_customer_id,country,...,invoice_hour,invoice_year_month,is_cancelled_invoice,is_negative_quantity,is_zero_quantity,is_missing_description,is_missing_customer_id,is_invalid_invoice_date,is_zero_or_negative_unit_price,is_duplicate_row
0,1,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,True,United Kingdom,...,8,2010-12,False,False,False,False,False,False,False,False
1,2,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,True,United Kingdom,...,8,2010-12,False,False,False,False,False,False,False,False
2,3,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,True,United Kingdom,...,8,2010-12,False,False,False,False,False,False,False,False
3,4,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,True,United Kingdom,...,8,2010-12,False,False,False,False,False,False,False,False
4,5,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,True,United Kingdom,...,8,2010-12,False,False,False,False,False,False,False,False


In [7]:
transactions.columns.tolist()

['raw_row_id',
 'invoice_no',
 'stock_code',
 'description',
 'quantity',
 'invoice_date',
 'unit_price',
 'customer_id',
 'has_customer_id',
 'country',
 'transaction_status',
 'line_revenue',
 'completed_sales_amount',
 'cancelled_amount',
 'invoice_year',
 'invoice_month',
 'invoice_day',
 'invoice_hour',
 'invoice_year_month',
 'is_cancelled_invoice',
 'is_negative_quantity',
 'is_zero_quantity',
 'is_missing_description',
 'is_missing_customer_id',
 'is_invalid_invoice_date',
 'is_zero_or_negative_unit_price',
 'is_duplicate_row']

## Table Size Review

This section checks the number of rows and columns in each cleaned output.

In [9]:
table_shapes = pd.DataFrame({
    "table_name": [
        "transactions_sample",
        "products",
        "customers",
        "cleaning_summary",
        "data_quality_issues_sample"
    ],
    "rows": [
        transactions.shape[0],
        products.shape[0],
        customers.shape[0],
        cleaning_summary.shape[0],
        data_quality_issues.shape[0]
    ],
    "columns": [
        transactions.shape[1],
        products.shape[1],
        customers.shape[1],
        cleaning_summary.shape[1],
        data_quality_issues.shape[1]
    ]
})

table_shapes

,table_name,rows,columns
0,transactions_sample,10000,27
1,products,4070,8
2,customers,4372,7
3,cleaning_summary,17,2
4,data_quality_issues_sample,10000,9


## Transaction Status Distribution

This section reviews how many cleaned transaction rows are classified as completed sales, cancelled/returns, or data issues.

In [10]:
transaction_status_counts = (
    transactions["transaction_status"]
    .value_counts()
    .reset_index()
)

transaction_status_counts.columns = ["transaction_status", "row_count"]

transaction_status_counts

,transaction_status,row_count
0,Completed,9854
1,Cancelled/Return,130
2,Data Issue,16


In [ ]:
## Revenue Summary from Sample

This section calculates completed sales and cancelled amount from the GitHub-safe cleaned sample.

The full KPI totals are calculated separately using the full local dataset and SQL outputs.

In [11]:
sample_revenue_summary = pd.DataFrame({
    "metric": [
        "sample_completed_sales_amount",
        "sample_cancelled_amount",
        "sample_net_sales_after_cancellations"
    ],
    "value": [
        round(transactions["completed_sales_amount"].sum(), 2),
        round(transactions["cancelled_amount"].sum(), 2),
        round(
            transactions["completed_sales_amount"].sum()
            - transactions["cancelled_amount"].sum(),
            2
        )
    ]
})

sample_revenue_summary

,metric,value
0,sample_completed_sales_amount,184248.33
1,sample_cancelled_amount,3580.58
2,sample_net_sales_after_cancellations,180667.75


In [ ]:
## Full Dataset KPI Summary

The full dataset KPI summary was generated from SQLite using SQL. This notebook loads the exported KPI CSV to show full-project metrics without loading large files into GitHub.

In [13]:
executive_kpi_file = sql_output_dir / "executive_kpi_summary.csv"
data_quality_kpi_file = sql_output_dir / "data_quality_kpi_summary.csv"

executive_kpi = pd.read_csv(executive_kpi_file)
data_quality_kpi = pd.read_csv(data_quality_kpi_file)

executive_kpi


,total_transaction_lines,completed_transaction_lines,cancelled_return_lines,data_issue_lines,completed_invoice_count,total_completed_sales_amount,total_cancelled_amount,net_sales_after_cancellations,average_order_value,average_completed_line_value,cancellation_amount_rate_percent,unique_product_count,known_customer_count,country_count,missing_customer_id_rows,missing_customer_id_rate_percent,duplicate_rows,duplicate_row_rate_percent
0,541909,530104,10624,1181,19960,10666684.54,896812.49,9769872.05,534.4,20.12,7.76,4070,4372,38,135080,24.93,10147,1.87


In [14]:
data_quality_kpi

,total_transaction_lines,missing_customer_id_rows,missing_customer_id_rate_percent,duplicate_rows,duplicate_row_rate_percent,zero_or_negative_unit_price_rows,missing_description_rows,data_issue_rows,data_quality_issue_records
0,541909,135080,24.93,10147,1.87,2517,1454,1181,169110


## Missing Customer ID Review

Missing customer IDs limit customer-level analysis such as segmentation, retention, repeat purchase behavior, and customer lifetime value.

In [15]:
missing_customer_summary = pd.DataFrame({
    "metric": [
        "sample_total_rows",
        "sample_missing_customer_id_rows",
        "sample_missing_customer_id_rate_percent"
    ],
    "value": [
        len(transactions),
        int((transactions["customer_id"] == "UNKNOWN").sum()),
        round((transactions["customer_id"] == "UNKNOWN").mean() * 100, 2)
    ]
})

missing_customer_summary

,metric,value
0,sample_total_rows,10000.00
1,sample_missing_customer_id_rows,2291.00
2,sample_missing_customer_id_rate_percent,22.91


In [ ]:
## Duplicate Row Review

Duplicate rows may create reporting risk if they inflate transaction counts or revenue totals. In this project, duplicate rows are flagged instead of silently removed.

In [19]:
duplicate_summary = (
    transactions
    .groupby("is_duplicate_row")
    .agg(
        row_count=("raw_row_id", "count"),
        completed_sales_amount=("completed_sales_amount", "sum")
    )
    .reset_index()
)

#duplicate_summary["completed_sales_amount"] = duplicate_summary["completed_sales_amount"].round(2)

duplicate_summary

,is_duplicate_row,row_count,completed_sales_amount
0,False,9619,182829.97
1,True,381,1418.36


## Data Quality Issue Type Review

The data quality issue log records row-level issues such as missing customer IDs, duplicate rows, negative quantities, cancelled invoices, missing descriptions, and invalid pricing values.

In [20]:
issue_type_summary = (
    data_quality_issues["issue_type"]
    .value_counts()
    .reset_index()
)

issue_type_summary.columns = ["issue_type", "issue_count"]

issue_type_summary

,issue_type,issue_count
0,Missing Customer ID,8260
1,Duplicate Row,670
2,Negative Quantity,422
3,Cancelled Invoice,355
4,Zero or Negative Unit Price,182
5,Missing Description,111


In [ ]:
## Initial Findings

Based on the cleaned data and full SQL KPI outputs:

- The cleaned dataset separates completed sales, cancelled/returns, and data issue rows.
- Full completed sales amount is approximately $10.67M.
- Full cancelled/return amount is approximately $896.8K.
- Missing customer IDs affect approximately 24.93% of transaction rows.
- Duplicate rows affect approximately 1.87% of transaction rows.
- Data quality issues are documented separately instead of being silently deleted.

## Professional Interpretation

The dataset is suitable for sales reporting, product analysis, country-level analysis, and operational review.

However, customer-level analysis should be handled carefully because a significant share of rows have missing customer IDs.